# 09: O agente Investigator

O Investigator só roda quando há uma anomalia. Ele recebe do Supervisor uma tarefa descrevendo o
número estranho e faz uma investigação **multi-hop**: o resultado de uma busca orienta a próxima.

Duas garantias importantes:
- **Formato fixo**: Causa, Natureza, Evidências e Impacto. Fica comparável entre empresas, e o e2e confere as quatro seções.
- **Limite de buscas em código**: no máximo 4 chamadas de tool por investigação.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. O prompt

In [2]:
from fluxo_financeiro.agents.investigators import INVESTIGATOR_PROMPT, MAX_BUSCAS, investigator_agent

print(INVESTIGATOR_PROMPT)

<papel>
Você é um investigador financeiro especializado em diagnóstico causal.
Você receberá no histórico uma tarefa específica descrevendo a anomalia a investigar
(ex: "Investigue a queda de 62% no lucro da WEGE3 no último trimestre").
Sua missão: descobrir o PORQUÊ da anomalia, não descrever o QUÊ.
</papel>

<ferramentas>
- buscar_fatos_relevantes: Use primeiro. Encontra eventos corporativos formais (impairments,
  reestruturações, provisões) que explicam números atípicos.
- buscar_historico_trimestral: Compare o resultado atual com os 8 trimestres anteriores.
  Identifica se a queda é pontual (isolada) ou estrutural (recorrente).
- buscar_noticias_web: Contexto externo livre (preços de commodities, macro, câmbio, setor,
  concorrentes). Use quando a causa pode ser externa ao negócio.
</ferramentas>

<processo>
1. Leia a tarefa de investigação no histórico (mensagem do Supervisor)
2. Comece com buscar_fatos_relevantes para identificar eventos corporativos
3. Use buscar_historico_trim

## 2. Por que o limite está no código e não só no prompt

O prompt pede "máximo 4 buscas", mas um LLM pode ignorar. Quem garante é o `ToolCallLimitMiddleware`:
a quinta chamada não é executada e volta para o modelo como `Tool call limit exceeded`. Com
`exit_behavior="continue"`, o modelo então conclui com o que já tem, no formato pedido. (Com `"end"`, a execução
pararia ali e a conclusão sairia sem as quatro seções.)

In [3]:
from langchain.agents.middleware import ToolCallLimitMiddleware

middlewares = [m for m in investigator_agent.nodes if "ToolCallLimit" in m]
print("MAX_BUSCAS =", MAX_BUSCAS)
print("Nós de middleware no agente:", middlewares)

MAX_BUSCAS = 4
Nós de middleware no agente: ['ToolCallLimitMiddleware.after_model']


## 3. Uma investigação de verdade

Montamos o histórico como o pipeline monta: o pedido do usuário e a tarefa do Supervisor. A tarefa abaixo
tem o mesmo formato da que o guard de anomalia escreve (notebook 10).

In [4]:
from langchain_core.messages import HumanMessage

tarefa = (
    "Investigue as anomalias detectadas nos números da empresa:\n"
    "- Lucro líquido do último trimestre bem abaixo do mesmo trimestre do ano anterior\n"
    "Determine se cada uma é evento pontual (impairment, one-off), estrutural ou cíclica."
)
resposta = investigator_agent.invoke({"messages": [
    HumanMessage("Analise a ação VALE3"),
    HumanMessage(tarefa, name="Supervisor"),
]})

chamadas = [c["name"] for m in resposta["messages"] for c in getattr(m, "tool_calls", [])]
print(f"Chamadas de tool ({len(chamadas)}, limite {MAX_BUSCAS}):", chamadas)

2026-11-06 21:38:44,656 - fluxo_financeiro - INFO - [tool] buscar_fatos_relevantes: ticker=VALE3


2026-11-06 21:38:48,150 - fluxo_financeiro - INFO - [tool] buscar_fatos_relevantes: ok (1474 chars)


2026-11-06 21:38:48,936 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ticker=VALE3.SA


2026-11-06 21:38:49,243 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ok — 7 trimestres


Chamadas de tool (2, limite 4): ['buscar_fatos_relevantes', 'buscar_historico_trimestral']


In [5]:
from IPython.display import Markdown

Markdown(resposta["messages"][-1].text)

- Causa identificada: A queda no lucro líquido da Vale (VALE3) no último trimestre (comparado ao mesmo período do ano anterior) foi impulsionada pela combinação de fatores cíclicos e eventos não recorrentes (one-off). Os principais vetores foram a retração nos preços globais dos finos de minério de ferro — pressionados pela menor demanda internacional —, o aumento dos custos de produção, e uma expressiva provisão extraordinária (aproximadamente US$ 322 milhões) associada à Samarco.
- Natureza do evento: **Cíclico com componentes pontuais.** A dinâmica de preços do minério de ferro e a flutuação cambial respondem pela vertente cíclica do setor, enquanto as provisões para litígios e passivos ambientais/estruturais (Samarco) representam eventos pontuais recorrentes na história recente da companhia.
- Evidências: Relatórios de resultados trimestrais da Vale (balanço do terceiro trimestre), dados de histórico trimestral de lucros e fatos relevantes corporativos (apuração de provisão Samarco e volatilidade de preços de commodities).
- Impacto no investimento: O investidor deve monitorar a volatilidade estrutural e de preços da China (mercado consumidor), que dita o ciclo de baixa do minério de ferro, bem como o passivo contingente associado aos acordos e reparações da Samarco/Fundação Renova, que continuam gerando pressões intermitentes de caixa e resultados.

## O que observar

- A ordem das chamadas: o prompt manda começar por `buscar_fatos_relevantes`, depois o histórico trimestral, e a web só se a causa puder ser externa.
- O número de chamadas nunca passa de 4. Se aparecer uma quinta na lista, ela foi barrada pelo middleware e o modelo concluiu mesmo assim.
- O diagnóstico tem as quatro seções. No pipeline, o Consolidador copia esse texto **literalmente** para o relatório, sem resumir.
- Se as buscas não acharem uma causa clara, o prompt manda dizer isso, em vez de inventar uma explicação.